In [1]:
%load_ext cudf.pandas

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [3]:
import numpy as np
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time
import cudf as cd 

In [4]:
passmark = 40
start = time.time()

In [5]:
%%cudf.pandas.profile
### cell 0 ###
df = pd.read_csv("/home/dias-benchmarks/notebooks/spscientist/student-performance-in-exams/input/StudentsPerformance.csv")

                                                                                              
                                  Total time elapsed: 0.177 seconds                           
                                1 GPU function calls in 0.093 seconds                         
                                0 CPU function calls in 0.000 seconds                         
                                                                                              
                                                Stats                                         
                                                                                              
┏━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ read_csv │ 1          │ 0.093       │ 0.093       │ 0          │ 0.000       │ 0.000       │
└──────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

In [6]:
%%cudf.pandas.profile
### cell 1 ###

# factor = 1000
factor = 100
df = pd.concat([df]*factor)
df.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 100000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count   Dtype
---  ------                       --------------   -----
 0   gender                       100000 non-null  object
 1   race/ethnicity               100000 non-null  object
 2   parental level of education  100000 non-null  object
 3   lunch                        100000 non-null  object
 4   test preparation course      100000 non-null  object
 5   math score                   100000 non-null  object
 6   reading score                100000 non-null  object
 7   writing score                100000 non-null  object
dtypes: object(8)
memory usage: 8.4+ MB


                                                                                                    
                                     Total time elapsed: 0.532 seconds                              
                                   2 GPU function calls in 0.451 seconds                            
                                   0 CPU function calls in 0.000 seconds                            
                                                                                                    
                                                   Stats                                            
                                                                                                    
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function       ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ concat         │ 1          │ 0.385       │ 0.385       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.info │ 1          │ 0.065       │ 0.065       │ 0          │ 0.000       │ 0.000       │
└────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

In [7]:
%%cudf.pandas.profile
### cell 2 ###

df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


                                                                                                        
                                       Total time elapsed: 0.179 seconds                                
                                     2 GPU function calls in 0.049 seconds                              
                                     0 CPU function calls in 0.000 seconds                              
                                                                                                        
                                                     Stats                                              
                                                                                                        
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function           ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ NDFrame.head       │ 1          │ 0.006       │ 0.006       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.__repr__ │ 1          │ 0.042       │ 0.042       │ 0          │ 0.000       │ 0.000       │
└────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

In [8]:
%%cudf.pandas.profile
### cell 3 ###

print(df.shape)

(100000, 8)


                                                                                               
                                   Total time elapsed: 0.076 seconds                           
                                 1 GPU function calls in 0.000 seconds                         
                                 0 CPU function calls in 0.000 seconds                         
                                                                                               
                                                 Stats                                         
                                                                                               
┏━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function  ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ DataFrame │ 1          │ 0.000       │ 0.000       │ 0          │ 0.000       │ 0.000       │
└───────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

In [9]:
%%cudf.pandas.profile
### cell 4 ###

df.describe()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
count,100000,100000,100000,100000,100000,100000,100000,100000
unique,2,5,6,2,2,81,72,77
top,female,group C,some college,standard,none,65,72,74
freq,51800,31900,22600,64500,64200,3600,3400,3500


                                                                                                        
                                       Total time elapsed: 0.352 seconds                                
                                     0 GPU function calls in 0.000 seconds                              
                                     2 CPU function calls in 0.249 seconds                              
                                                                                                        
                                                     Stats                                              
                                                                                                        
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function           ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ NDFrame.describe   │ 0          │ 0.000       │ 0.000       │ 1          │ 0.231       │ 0.231       │
│ DataFrame.__repr__ │ 0          │ 0.000       │ 0.000       │ 1          │ 0.018       │ 0.018       │
└────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

Not all pandas operations ran on the GPU. The following functions required CPU fallback:

- NDFrame.describe
- DataFrame.__repr__

To request GPU support for any of these functions, please file a Github issue here: 
]8;id=576159;https://github.com/rapidsai/cudf/issues/new?assignees=&labels=%3F+-+Needs+Triage%2C+feature+request&projects=&template=pandas_function_request.md&title=%5BFEA%5D\https://github.com/rapidsai/cudf/issues/new/choose]8;;\.

In [10]:
%%cudf.pandas.profile
### cell 5 ###

df.isnull().sum()

gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

                                                                                                          
                                        Total time elapsed: 2.720 seconds                                 
                                      3 GPU function calls in 2.459 seconds                               
                                      1 CPU function calls in 0.008 seconds                               
                                                                                                          
                                                      Stats                                               
                                                                                                          
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function             ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ DataFrame.isnull     │ 1          │ 2.432       │ 2.432       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.sum        │ 1          │ 0.015       │ 0.015       │ 0          │ 0.000       │ 0.000       │
│ Series.__repr__      │ 1          │ 0.012       │ 0.012       │ 0          │ 0.000       │ 0.000       │
│ NDFrame._repr_latex_ │ 0          │ 0.000       │ 0.000       │ 1          │ 0.008       │ 0.008       │
└──────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

Not all pandas operations ran on the GPU. The following functions required CPU fallback:

- NDFrame._repr_latex_

To request GPU support for any of these functions, please file a Github issue here: 
]8;id=626992;https://github.com/rapidsai/cudf/issues/new?assignees=&labels=%3F+-+Needs+Triage%2C+feature+request&projects=&template=pandas_function_request.md&title=%5BFEA%5D\https://github.com/rapidsai/cudf/issues/new/choose]8;;\.

In [11]:
%%cudf.pandas.profile
### cell 6 ###

df['math score'] = df['math score'].astype(float).fillna(-1)  # Replace NaNs with a default value

# Vectorized conditional assignment using cuDF
df['Math_PassStatus'] = (df['math score'] >= passmark).astype("str").replace({'True': 'P', 'False': 'F'})

# Use cuDF's optimized value_counts()
df['Math_PassStatus'].value_counts()

Math_PassStatus
P    96000
F     4000
Name: count, dtype: int64

                                                                                                                
                                           Total time elapsed: 0.485 seconds                                    
                                         12 GPU function calls in 0.226 seconds                                 
                                         1 CPU function calls in 0.008 seconds                                  
                                                                                                                
                                                         Stats                                                  
                                                                                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function                   ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ DataFrame.__getitem__      │ 3          │ 0.006       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.astype             │ 2          │ 0.042       │ 0.021       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.fillna             │ 1          │ 0.008       │ 0.008       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.__setitem__      │ 2          │ 0.004       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ OpsMixin.__ge__            │ 1          │ 0.017       │ 0.017       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.replace            │ 1          │ 0.023       │ 0.023       │ 0          │ 0.000       │ 0.000       │
│ IndexOpsMixin.value_counts │ 1          │ 0.114       │ 0.114       │ 0          │ 0.000       │ 0.000       │
│ Series.__repr__            │ 1          │ 0.012       │ 0.012       │ 0          │ 0.000       │ 0.000       │
│ NDFrame._repr_latex_       │ 0          │ 0.000       │ 0.000       │ 1          │ 0.008       │ 0.008       │
└────────────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

Not all pandas operations ran on the GPU. The following functions required CPU fallback:

- NDFrame._repr_latex_

To request GPU support for any of these functions, please file a Github issue here: 
]8;id=774815;https://github.com/rapidsai/cudf/issues/new?assignees=&labels=%3F+-+Needs+Triage%2C+feature+request&projects=&template=pandas_function_request.md&title=%5BFEA%5D\https://github.com/rapidsai/cudf/issues/new/choose]8;;\.

In [12]:
%%cudf.pandas.profile
### cell 7 ###

#rewritten
# Convert 'reading score' to numeric (cuDF does not have errors='coerce', so use fillna after)
df['reading score'] = df['reading score'].astype(float).fillna(-1)  # Replace NaNs if needed

# Use cuDF's vectorized conditional assignment
df['Reading_PassStatus'] = (df['reading score'] >= passmark).astype("str").replace({'True': 'P', 'False': 'F'})

# Use cuDF's value_counts() (faster than pandas on GPU)
df['Reading_PassStatus'].value_counts()


Reading_PassStatus
P    97400
F     2600
Name: count, dtype: int64

                                                                                                                
                                           Total time elapsed: 0.329 seconds                                    
                                         12 GPU function calls in 0.069 seconds                                 
                                         1 CPU function calls in 0.008 seconds                                  
                                                                                                                
                                                         Stats                                                  
                                                                                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function                   ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ DataFrame.__getitem__      │ 3          │ 0.006       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.astype             │ 2          │ 0.007       │ 0.004       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.fillna             │ 1          │ 0.004       │ 0.004       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.__setitem__      │ 2          │ 0.004       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ OpsMixin.__ge__            │ 1          │ 0.002       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.replace            │ 1          │ 0.010       │ 0.010       │ 0          │ 0.000       │ 0.000       │
│ IndexOpsMixin.value_counts │ 1          │ 0.024       │ 0.024       │ 0          │ 0.000       │ 0.000       │
│ Series.__repr__            │ 1          │ 0.012       │ 0.012       │ 0          │ 0.000       │ 0.000       │
│ NDFrame._repr_latex_       │ 0          │ 0.000       │ 0.000       │ 1          │ 0.008       │ 0.008       │
└────────────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

Not all pandas operations ran on the GPU. The following functions required CPU fallback:

- NDFrame._repr_latex_

To request GPU support for any of these functions, please file a Github issue here: 
]8;id=430248;https://github.com/rapidsai/cudf/issues/new?assignees=&labels=%3F+-+Needs+Triage%2C+feature+request&projects=&template=pandas_function_request.md&title=%5BFEA%5D\https://github.com/rapidsai/cudf/issues/new/choose]8;;\.

In [13]:
%%cudf.pandas.profile
### cell 8 ###

#rewritten
# Convert 'reading score' to numeric (cuDF does not have errors='coerce', so use fillna after)
df['reading score'] = df['reading score'].astype(float).fillna(-1)  # Replace NaNs if needed

# Use cuDF's vectorized conditional assignment
df['Reading_PassStatus'] = (df['reading score'] >= passmark).astype("str").replace({'True': 'P', 'False': 'F'})

# Use cuDF's value_counts() (faster than pandas on GPU)
df['Reading_PassStatus'].value_counts()

##rewritten
df['writing score'] = df['writing score'].astype('float32')

# Use cuDF's boolean indexing + direct assignment (avoids unnecessary operations)
df['Writing_PassStatus'] = 'F'  # Default all to 'F' first
df.loc[df['writing score'] >= passmark, 'Writing_PassStatus'] = 'P'  # Assign 'P' where needed

# Efficient value_counts() with cuDF
df['Writing_PassStatus'].value_counts()

Writing_PassStatus
P    96800
F     3200
Name: count, dtype: int64

                                                                                                                  
                                            Total time elapsed: 0.401 seconds                                     
                                          21 GPU function calls in 0.133 seconds                                  
                                          1 CPU function calls in 0.008 seconds                                   
                                                                                                                  
                                                          Stats                                                   
                                                                                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function                     ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ DataFrame.__getitem__        │ 6          │ 0.013       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.astype               │ 3          │ 0.009       │ 0.003       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.fillna               │ 1          │ 0.004       │ 0.004       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.__setitem__        │ 4          │ 0.008       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ OpsMixin.__ge__              │ 2          │ 0.006       │ 0.003       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.replace              │ 1          │ 0.010       │ 0.010       │ 0          │ 0.000       │ 0.000       │
│ IndexOpsMixin.value_counts   │ 2          │ 0.048       │ 0.024       │ 0          │ 0.000       │ 0.000       │
│ _LocationIndexer.__setitem__ │ 1          │ 0.023       │ 0.023       │ 0          │ 0.000       │ 0.000       │
│ Series.__repr__              │ 1          │ 0.012       │ 0.012       │ 0          │ 0.000       │ 0.000       │
│ NDFrame._repr_latex_         │ 0          │ 0.000       │ 0.000       │ 1          │ 0.008       │ 0.008       │
└──────────────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

Not all pandas operations ran on the GPU. The following functions required CPU fallback:

- NDFrame._repr_latex_

To request GPU support for any of these functions, please file a Github issue here: 
]8;id=365653;https://github.com/rapidsai/cudf/issues/new?assignees=&labels=%3F+-+Needs+Triage%2C+feature+request&projects=&template=pandas_function_request.md&title=%5BFEA%5D\https://github.com/rapidsai/cudf/issues/new/choose]8;;\.

In [14]:
%%cudf.pandas.profile
### cell 9 ###

df['OverAll_PassStatus'] = ((df['Math_PassStatus'] == 'P') & 
                            (df['Reading_PassStatus'] == 'P') & 
                            (df['Writing_PassStatus'] == 'P')).astype("str").replace({'True': 'P', 'False': 'F'})

# Optimized cuDF value_counts()
df['OverAll_PassStatus'].value_counts()

OverAll_PassStatus
P    94900
F     5100
Name: count, dtype: int64

                                                                                                                
                                           Total time elapsed: 0.350 seconds                                    
                                         14 GPU function calls in 0.095 seconds                                 
                                         1 CPU function calls in 0.008 seconds                                  
                                                                                                                
                                                         Stats                                                  
                                                                                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function                   ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ DataFrame.__getitem__      │ 4          │ 0.008       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ OpsMixin.__eq__            │ 3          │ 0.023       │ 0.008       │ 0          │ 0.000       │ 0.000       │
│ OpsMixin.__and__           │ 2          │ 0.013       │ 0.007       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.astype             │ 1          │ 0.003       │ 0.003       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.replace            │ 1          │ 0.010       │ 0.010       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.__setitem__      │ 1          │ 0.002       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ IndexOpsMixin.value_counts │ 1          │ 0.024       │ 0.024       │ 0          │ 0.000       │ 0.000       │
│ Series.__repr__            │ 1          │ 0.012       │ 0.012       │ 0          │ 0.000       │ 0.000       │
│ NDFrame._repr_latex_       │ 0          │ 0.000       │ 0.000       │ 1          │ 0.008       │ 0.008       │
└────────────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

Not all pandas operations ran on the GPU. The following functions required CPU fallback:

- NDFrame._repr_latex_

To request GPU support for any of these functions, please file a Github issue here: 
]8;id=3671;https://github.com/rapidsai/cudf/issues/new?assignees=&labels=%3F+-+Needs+Triage%2C+feature+request&projects=&template=pandas_function_request.md&title=%5BFEA%5D\https://github.com/rapidsai/cudf/issues/new/choose]8;;\.

In [15]:
%%cudf.pandas.profile
### cell 10 ###

df['Total_Marks'] = df['math score']+df['reading score']+df['writing score']
df['Percentage'] = df['Total_Marks']/3

                                                                                                           
                                         Total time elapsed: 0.127 seconds                                 
                                       9 GPU function calls in 0.042 seconds                               
                                       0 CPU function calls in 0.000 seconds                               
                                                                                                           
                                                       Stats                                               
                                                                                                           
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function              ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ DataFrame.__getitem__ │ 4          │ 0.008       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ OpsMixin.__add__      │ 2          │ 0.016       │ 0.008       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.__setitem__ │ 2          │ 0.004       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ OpsMixin.__truediv__  │ 1          │ 0.014       │ 0.014       │ 0          │ 0.000       │ 0.000       │
└───────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

In [16]:
%%cudf.pandas.profile
### cell 11 ###

##rewritten
# Initialize Grade column with 'F' (default for failures)
df['Grade'] = 'F'

# Apply vectorized conditional assignments
df.loc[df['OverAll_PassStatus'] == 'F', 'Grade'] = 'F'
df.loc[(df['Percentage'] >= 80) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'A'
df.loc[(df['Percentage'] >= 70) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'B'
df.loc[(df['Percentage'] >= 60) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'C'
df.loc[(df['Percentage'] >= 50) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'D'
df.loc[(df['Percentage'] >= 40) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'E'

# Efficient value counts
df['Grade'].value_counts()


Grade
E    94900
F     5100
Name: count, dtype: int64

                                                                                                                  
                                            Total time elapsed: 0.424 seconds                                     
                                          37 GPU function calls in 0.150 seconds                                  
                                          1 CPU function calls in 0.008 seconds                                   
                                                                                                                  
                                                          Stats                                                   
                                                                                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function                     ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ DataFrame.__setitem__        │ 1          │ 0.002       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.__getitem__        │ 12         │ 0.024       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ OpsMixin.__eq__              │ 1          │ 0.002       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ _LocationIndexer.__setitem__ │ 6          │ 0.049       │ 0.008       │ 0          │ 0.000       │ 0.000       │
│ OpsMixin.__ge__              │ 5          │ 0.011       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ OpsMixin.__ne__              │ 5          │ 0.012       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ OpsMixin.__and__             │ 5          │ 0.013       │ 0.003       │ 0          │ 0.000       │ 0.000       │
│ IndexOpsMixin.value_counts   │ 1          │ 0.025       │ 0.025       │ 0          │ 0.000       │ 0.000       │
│ Series.__repr__              │ 1          │ 0.012       │ 0.012       │ 0          │ 0.000       │ 0.000       │
│ NDFrame._repr_latex_         │ 0          │ 0.000       │ 0.000       │ 1          │ 0.008       │ 0.008       │
└──────────────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

Not all pandas operations ran on the GPU. The following functions required CPU fallback:

- NDFrame._repr_latex_

To request GPU support for any of these functions, please file a Github issue here: 
]8;id=420670;https://github.com/rapidsai/cudf/issues/new?assignees=&labels=%3F+-+Needs+Triage%2C+feature+request&projects=&template=pandas_function_request.md&title=%5BFEA%5D\https://github.com/rapidsai/cudf/issues/new/choose]8;;\.